In [789]:
# !pip install elasticsearch sentence-transformers

In [790]:
from elasticsearch import Elasticsearch, helpers
from dotenv import load_dotenv
import json
import pandas as pd
import os

In [791]:
from IPython.display import HTML

In [792]:
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings

In [793]:
''' pip install python-dotenv'''
# load_dotenv() # will search for .env file in local folder and load variables
# Reload the variables from your .env file, overriding existing ones
load_dotenv("../.env", override=True)

True

### __Elasticsearch Vector Recommendation __
* Vector search leverages machine learning (ML) to capture the meaning and context of unstructured data, including text and images, transforming it into a numeric representatio
* Building an Elasticsearch vector recommendation engine involves converting items into vector embeddings and querying them using a k-Nearest Neighbor (kNN) search.
* This allows your system to recommend items that are semantically similar in taste, context, or features

In [794]:
def get_headers():
    ''' Elasticsearch Header '''
    return {
            'Content-type': 'application/json', 
            # 'Authorization' : '{}'.format(os.getenv('BASIC_AUTH')),
            # 'Connection': 'close'
    }

In [795]:
def get_es_instance(host):
    es_client = Elasticsearch(hosts="{}".format(host), headers=get_headers(), timeout=5,  verify_certs=False)
    return es_client

In [796]:
# Create an instance
es = get_es_instance(f"http://{os.getenv('ES_VECTOR')}")

/tmp/ipykernel_37726/3222679292.py:2: DeprecationWarning: The 'timeout' parameter is deprecated in favor of 'request_timeout'
  es_client = Elasticsearch(hosts="{}".format(host), headers=get_headers(), timeout=5,  verify_certs=False)


In [797]:
# Fetch cluster info
info = es.info()
print("Cluster Version:", info['version']['number'])

Cluster Version: 8.7.1


### __Set Up the Index Mapping__
* First, configure your Elasticsearch index with a dense_vector field to hold the embeddings and an index mapping so you can run fast, approximate nearest neighbor (ANN) searches.

In [798]:
INDEX_NAME = "item_recommendations"

In [799]:
# Define mapping configuration
index_mapping = {
    "mappings": {
        "properties": {
           "item_name": { "type": "text" },
           "category": { "type": "keyword" },
           "item_vector": {
                "type": "dense_vector",
                "dims": 512,
                "index": True,
                "similarity": "cosine"
            }
        }
    }
}

In [800]:
# Delete the index if recreating it to clear old data
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

''' Create Index '''
es.indices.create(index=INDEX_NAME, body=index_mapping)
print(f"Index '{INDEX_NAME}' created successfully.")

Index 'item_recommendations' created successfully.


### __Generate and Store Embeddings__
* To recommend items, you need vector representations for each one. You can use external models (like those from Hugging Face, OpenAI, or Cohere) to convert item descriptions into arrays of floats and ingest them via the Elasticsearch Python Clien

In [801]:
# Initialize the embedding model

# 로컬에 저장된 모델 경로 불러오기
model_path = "../huggingfase_model/distiluse-base-multilingual-cased-v1/"

# embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embedding_model = HuggingFaceEmbeddings(model_name=model_path)
print(f"Embeddings: {embedding_model}")

Embeddings: model_name='../huggingfase_model/distiluse-base-multilingual-cased-v1/' cache_folder=None model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


In [802]:
# es = Elasticsearch("http://localhost:9200")
# model = SentenceTransformer("all-MiniLM-L6-v2")

# Example item to recommend
item_text = "Wireless noise-canceling headphones with deep bass"
# item_vector = model.encode(item_text).tolist()
''' Huggingface Embedding Model '''
item_vector = embedding_model.embed_query(item_text)

In [803]:
embedding[:3]

[-0.032983243465423584, 0.06595643609762192, -0.09756933152675629]

In [804]:
# Index the document
doc = {
    "id" : 1,
    "item_name": "Premium Bass Headphones",
    "category": "Electronics",
    "item_vector": item_vector
}

# Store documentation
resp = es.index(index=INDEX_NAME, id=doc["id"], body=doc)

In [805]:
# Refresh the index to make data immediately searchable
es.indices.refresh(index=INDEX_NAME)

ObjectApiResponse({'_shards': {'total': 2, 'successful': 1, 'failed': 0}})

In [806]:
print(json.dumps(resp.body, indent=2))

{
  "_index": "item_recommendations",
  "_id": "1",
  "_version": 1,
  "result": "created",
  "_shards": {
    "total": 2,
    "successful": 1,
    "failed": 0
  },
  "_seq_no": 0,
  "_primary_term": 1
}


### __Query for Recommendations__
* When a user views an item, you can use the vector representation of that item as your query_vector to find and return the closest matches in the database

In [807]:
# Assuming you have the vector of the current item being viewed
# query_vector = model.encode("Wireless noise-canceling headphones").tolist()
query_vector = embedding_model.embed_query("Wireless noise-canceling headphones")

response = es.search(
    index=INDEX_NAME,
    knn={
        "field": "item_vector",
        "query_vector": query_vector,
        "k": 5,                     # Return the top 5 closest items
        "num_candidates": 50        # Number of nearest neighbors to consider per shard
    },
    # source=["item_name", "category"]
)

In [808]:
data = [
        {
            "id" : doc.get("_id"),
            "text_field": doc.get("_source").get("item_name"),
            "_score" : f"{doc.get('_score'):.4f}"   # Use f-string 
        } 
        for doc in response["hits"]["hits"]
]

df = pd.DataFrame(data)
df

,id,text_field,_score
0,1,Premium Bass Headphones,0.9101


In [809]:
for hit in response["hits"]["hits"]:
    print(f"Recommended: {hit['_source']['item_name']} (Score: {hit['_score']})")

Recommended: Premium Bass Headphones (Score: 0.91009086)


### __Advanced: Hybrid Search and Filtering__
* To build a production-grade recommender, combine semantic vector search with keyword matching (BM25) and business rules. For example, __<i>you can filter recommendations so that it only recommends items that are in_stock = true</i>__